In [1]:
import pandas as pd
messages=pd.read_csv('smsspanclassification/SMSSpamCollection',
                    sep='\t',names=["label","message"])

In [2]:
messages

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [3]:
messages.label.value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [4]:
import re
import nltk
from nltk.corpus import stopwords
# from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [5]:
# ps = PorterStemmer()

In [6]:
corpus = []
for i in range(0, len(messages)):
    review = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    review = review.lower()
    review = review.split()
    review = [lemmatizer.lemmatize(word) for word in review if word not in stopwords.words('english')]
    review = ' '.join(review)
    corpus.append(review)

In [7]:
corpus

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply',
 'u dun say early hor u c already say',
 'nah think go usf life around though',
 'freemsg hey darling week word back like fun still tb ok xxx std chgs send rcv',
 'even brother like speak treat like aid patent',
 'per request melle melle oru minnaminunginte nurungu vettam set callertune caller press copy friend callertune',
 'winner valued network customer selected receivea prize reward claim call claim code kl valid hour',
 'mobile month u r entitled update latest colour mobile camera free call mobile update co free',
 'gonna home soon want talk stuff anymore tonight k cried enough today',
 'six chance win cash pound txt csh send cost p day day tsandcs apply reply hl info',
 'urgent week free membership prize jackpot txt word claim c www dbuk net lccltd pobox ldnw rw'

## Create Bag Of Words

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

In [24]:
cv = CountVectorizer(max_features=1000, binary=True)
X = cv.fit_transform(corpus).toarray()

In [25]:
X.shape

(5572, 1000)

In [26]:
print(cv.vocabulary_)

{'go': np.int64(327), 'point': np.int64(644), 'crazy': np.int64(168), 'available': np.int64(48), 'great': np.int64(340), 'world': np.int64(972), 'la': np.int64(432), 'got': np.int64(336), 'wat': np.int64(932), 'ok': np.int64(588), 'lar': np.int64(437), 'wif': np.int64(951), 'free': np.int64(295), 'entry': np.int64(243), 'wkly': np.int64(963), 'comp': np.int64(145), 'win': np.int64(954), 'cup': np.int64(171), 'final': np.int64(278), 'st': np.int64(799), 'may': np.int64(502), 'text': np.int64(842), 'receive': np.int64(684), 'question': np.int64(667), 'std': np.int64(807), 'txt': np.int64(890), 'rate': np.int64(672), 'apply': np.int64(35), 'dun': np.int64(223), 'say': np.int64(725), 'early': np.int64(227), 'already': np.int64(22), 'nah': np.int64(556), 'think': np.int64(849), 'usf': np.int64(909), 'life': np.int64(459), 'around': np.int64(39), 'though': np.int64(854), 'freemsg': np.int64(296), 'hey': np.int64(368), 'week': np.int64(939), 'word': np.int64(969), 'back': np.int64(57), 'like'

## spam classification using BoW

In [27]:
import pandas as pd

In [28]:
y = pd.get_dummies(messages['label'])
y.head()

,ham,spam
0,True,False
1,True,False
2,False,True
3,True,False
4,True,False


In [29]:
y=y.iloc[:,0].values

In [30]:
y

array([ True,  True, False, ...,  True,  True,  True], shape=(5572,))

In [31]:
y=y.astype(int)

In [32]:
y

array([1, 1, 0, ..., 1, 1, 1], shape=(5572,))

In [33]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.30)

In [34]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB()
classifier.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [35]:
y_pred = classifier.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report
score = accuracy_score(y_test, y_pred)
print(f'Accuracy: {round(score*100,2)}%')
print(classification_report(y_test, y_pred))

Accuracy: 97.85%
              precision    recall  f1-score   support

           0       0.93      0.92      0.92       229
           1       0.99      0.99      0.99      1443

    accuracy                           0.98      1672
   macro avg       0.96      0.95      0.95      1672
weighted avg       0.98      0.98      0.98      1672



## Spam classification using TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=1000, binary=True)
X2 = tfidf.fit_transform(corpus).toarray()

In [37]:
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size = 0.30)

In [38]:
spam_model_tfidf = MultinomialNB()
spam_model_tfidf.fit(X2_train, y2_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [39]:
y_pred2 = spam_model_tfidf.predict(X2_test)

In [41]:
score2 = accuracy_score(y2_test, y_pred2)
print(f'Accuracy: {round(score2*100,2)}%')
print(classification_report(y2_test, y_pred2))

Accuracy: 97.43%
              precision    recall  f1-score   support

           0       0.91      0.89      0.90       227
           1       0.98      0.99      0.99      1445

    accuracy                           0.97      1672
   macro avg       0.95      0.94      0.94      1672
weighted avg       0.97      0.97      0.97      1672

